<!-- REFACTOR-COMMENT: POST_PROCESSING -->
### Refactor Comment (Post-Processing)
- **Purpose**: Compare distortion-based subsidy diagnostics across scenarios for one run.
- **Primary inputs**: `runs/<run_name>/*/subsidies_distortion.csv`.
- **Primary outputs**: `img/subsidies_gap.png` and per-scenario subsidy-vs-distortion scatters.
- **Refactor role**: Keep distortion diagnostics separate from subsidy-allocation and policy-gap summaries.


<!-- NOTEBOOK-WORKFLOW -->
## Notebook Workflow
### What This Notebook Does
1. Resolve a policy-assessment run folder and load all scenario `subsidies_distortion.csv` tables.
2. Build `Subsidies Gap = Distortion - Subsidies` for each scenario.
3. Generate distortion-based distribution and scatter diagnostics.

### Inputs
- `project/analysis/post_processing/policy_assessment/runs/<run_name>/*/subsidies_distortion.csv`

### Outputs
- `project/analysis/post_processing/policy_assessment/runs/<run_name>/img/subsidies_gap.png`
- `project/analysis/post_processing/policy_assessment/runs/<run_name>/img/subsidies_distortion_<scenario>.png`

### How To Reuse
- Update `run_name`, optional filters, and scenario labels in the configuration cell.


# Analyze subsidy-distortion runs


In [ ]:
from pathlib import Path

import sys
sys.path.insert(0, "../../../..")

from project.analysis.post_processing.shared.io_runs import load_child_csv_bundle
from project.analysis.post_processing.shared.paths import resolve_run
from project.analysis.post_processing.shared.transforms import keep_scenarios, add_subsidies_gap
from project.analysis.post_processing.shared.plots import (
    plot_subsidies_gap_boxplot,
    plot_subsidy_vs_distortion,
)
from project.utils import make_scatter_plot


In [ ]:
# Run selection
run_name = "subsidies_distortions"
selected_scenarios = None  # Example: ["Package2024", "OptimalSubsidies"]

# Scenario display names used in plots
scenario_renames = {"Reference": "Package2024"}
scenario_title_labels = {
    "Package2024": "Package 2024",
    "OptimalSubsidies": "Optimal subsidies",
}

remove_technologies = [
    "Electricity-Direct electric",
    "Natural gas-Performance boiler",
    "Oil fuel-Performance boiler",
    "Wood fuel-Performance boiler",
    "Heating-District heating",
]


In [ ]:
run_folder = resolve_run("policy_assessment", run_name)
output_folder = run_folder / "img"
output_folder.mkdir(parents=True, exist_ok=True)

print(f"Run folder: {run_folder}")
print(f"Output folder: {output_folder}")

distortion_bundle = load_child_csv_bundle(run_folder, "subsidies_distortion.csv", index_col=None)
if not distortion_bundle:
    raise FileNotFoundError(f"No subsidies_distortion.csv files found in {run_folder}")

print(f"Loaded distortion tables: {len(distortion_bundle)}")


In [ ]:
distortion_bundle = keep_scenarios(distortion_bundle, selected_scenarios)

if selected_scenarios:
    print(f"Filtered scenarios: {list(distortion_bundle.keys())}")

if not distortion_bundle:
    raise ValueError("Scenario filter removed all data. Update `selected_scenarios`.")

if scenario_renames:
    distortion_bundle = {scenario_renames.get(name, name): table for name, table in distortion_bundle.items()}

def display_scenario(name):
    return scenario_title_labels.get(name, name)

comparison_title = " vs. ".join(display_scenario(scenario) for scenario in distortion_bundle)
distortion_bundle = add_subsidies_gap(distortion_bundle)


In [ ]:
plot_subsidies_gap_boxplot(
    distortion_bundle,
    output_folder / "subsidies_gap.png",
    title=f"Distribution of distortion-based subsidy gaps across dwelling segments: {comparison_title}",
)


In [ ]:
plot_subsidy_vs_distortion(
    bundle=distortion_bundle,
    output_folder=output_folder,
    technology_filter=remove_technologies,
    make_scatter_fn=make_scatter_plot,
    title_template="Subsidies and distortion by dwelling technology: {scenario}",
    x_label="",
)
